# CRIVO — Exportar a v4 F para NumPy (chat e app)

Converte os pesos da fase 5 v4, variante F (SHA `1d47312f…`), para o formato NumPy do site (float16). Confere se o NumPy dá o mesmo resultado que o PyTorch e divide o arquivo em partes de até 3,5 MB no Drive, para entrar no repositório.

**A v4 F foi reprovada na catraca** (blind v3 63,5% < 66,2%; urgência no blind v3 71%). Ela vai para o chat por decisão do criador, registrada como reprovada; "peito apertado" passou para o protocolo de crise.

**Não precisa de GPU.** Ambiente de execução → Executar tudo (uns 2 minutos). Resultado em `MyDrive/CRIVO/fase5v4f-numpy/`.

In [ ]:
# Célula 0 — Drive e pesos da v4 F
import os, sys, json, shutil, subprocess, hashlib
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
D_SHA = "1d47312f3d99cfc2b20cf010414984367fd8d9b2566ebfcb14ee21dda11732d5"
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
ORIGEM = None
for crivo in candidatas:
    for p in sorted((crivo / "transformer-17m-fase5-v4").glob("fase5v4_*/pesos_F.pt")):
        if sha(p) == D_SHA:
            ORIGEM, CRIVO = p, crivo
            break
    if ORIGEM:
        break
if ORIGEM is None:
    raise SystemExit("ERRO: não achei pesos_F.pt (sha 1d47312f…) em CRIVO/transformer-17m-fase5-v4/. "
                     "Rode na conta onde a v4 treinou (hr87676620).")
LOCAL = Path("/content/fase5v4_f")
shutil.rmtree(LOCAL, ignore_errors=True)
LOCAL.mkdir()
shutil.copyfile(ORIGEM, LOCAL / "pesos.pt")
shutil.copyfile(ORIGEM.parent / "tokenizer.json", LOCAL / "tokenizer.json")
print("pesos da v4 F:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — código do CRIVO e exportação com conferência NumPy × PyTorch
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
subprocess.run([sys.executable, "scripts/exportar_pontuador.py", "--origem", str(LOCAL), "--destino", str(LOCAL)],
               cwd=REPO, check=True)
NPZ = LOCAL / "pesos_numpy.npz"
print("npz:", round(NPZ.stat().st_size / 1e6, 1), "MB | sha256", sha(NPZ), flush=True)


In [ ]:
# Célula 2 — partes de até 3,5 MB no Drive (o conector do Drive não entrega arquivos maiores)
SAIDA = CRIVO / "fase5v4f-numpy"
SAIDA.mkdir(parents=True, exist_ok=True)
for antigo in SAIDA.glob("parte_*"):
    antigo.unlink()
dados = NPZ.read_bytes()
TAM = 3584 * 1024
partes = []
for i in range(0, len(dados), TAM):
    nome = "parte_%02d.bin" % (i // TAM)
    (SAIDA / nome).write_bytes(dados[i:i + TAM])
    partes.append({"nome": nome, "sha256": hashlib.sha256(dados[i:i + TAM]).hexdigest()})
manifesto = {"pesos_origem_sha256": D_SHA, "npz_sha256": sha(NPZ), "npz_bytes": len(dados), "partes": partes,
             "tokenizer_sha256": sha(LOCAL / "tokenizer.json")}
(SAIDA / "manifesto_fase5v4f.json").write_text(json.dumps(manifesto, indent=1))
print(json.dumps(manifesto, indent=1))
print("\nPRONTO: me avise que eu pego as partes no Drive.", flush=True)
